In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET

In [3]:
croot = "D:/FEWS-PROJECTS/WSDD-WIS/FEWS/Config"
config_root = Path(croot)

In [4]:
ns = {'fews': 'http://www.wldelft.nl/fews', 'xsi': 'http://www.w3.org/2001/XMLSchema-instance'}

In [ ]:
tree = ET.parse(config_root/"RegionConfigFiles/WorkflowDescriptors.xml")
root = tree.getroot()

In [ ]:
for wfd in root.findall('fews:workflowDescriptor', ns):
    wf_id = wfd.attrib["id"]
    description = wfd.find('fews:description', ns).text
    try:
        workflowFileName = wfd.find('fews:workflowFileName', ns).text
    except AttributeError:
        workflowFileName = wf_id
    con.execute("INSERT INTO wfs VALUES (?, ?, ?)", [wf_id, description, workflowFileName])

    con.execute("INSERT INTO wfd_props VALUES (?, ?, ?)", [wf_id, "workflowFileName", workflowFileName])
    
    # optional properties
    for props in wfd.findall('fews:properties', ns):
        for prop in props.findall('fews:string', ns):
            con.execute("INSERT INTO wfd_props VALUES (?, ?, ?)", [wf_id, prop.attrib["key"], prop.attrib["value"]])

In [ ]:
con.execute("SELECT id, wf_filename FROM wfs")
for row in con.fetchall():
    wf_filename = "{}.xml".format(row[1])
    for f in config_root.glob(f"**/{wf_filename}"):
        wf_filepath = f
    print("{} -> {} --> {}".format(row[0], wf_filename, wf_filepath.absolute()))


In [5]:
wf_tree = ET.parse(config_root / "WorkflowFiles/RTCTools/wf_RunRtcRerengAI.xml")
wf_root = wf_tree.getroot()

In [ ]:
wf_root.attrib

In [12]:
for prop in wf_root.find('fews:properties', ns):
    print("{} --> {}".format(prop.attrib["key"], prop.attrib["value"]))

forecast --> RerengAI
LeadTime --> 15
FeedBackFactor --> 1


In [13]:
for x in wf_root.findall('fews:activity', ns):
    print(x.attrib)
    print(x)

{'logStartedAsDebug': 'true', 'logFinishedAsDebug': 'true'}
<Element '{http://www.wldelft.nl/fews}activity' at 0x0000025C66832ED0>
{'logStartedAsDebug': 'true', 'logFinishedAsDebug': 'true'}
<Element '{http://www.wldelft.nl/fews}activity' at 0x0000025C66833010>
{'logStartedAsDebug': 'true', 'logFinishedAsDebug': 'false'}
<Element '{http://www.wldelft.nl/fews}activity' at 0x0000025C668330B0>
{'logStartedAsDebug': 'true', 'logFinishedAsDebug': 'false'}
<Element '{http://www.wldelft.nl/fews}activity' at 0x0000025C668331F0>
{'logStartedAsDebug': 'true', 'logFinishedAsDebug': 'true'}
<Element '{http://www.wldelft.nl/fews}activity' at 0x0000025C66833380>


In [ ]:
for wfd in root.findall('fews:workflowDescriptor', ns):
    description = wfd.find('fews:description', ns).text
    try:
        workflowFileName = wfd.find('fews:workflowFileName', ns).text
    except AttributeError:
        workflowFileName = wfd.attrib["id"]
    print(description, workflowFileName)

In [ ]:
con.close()
Path.unlink(dbpath)